<h2>Import requirements

In [48]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
import tensorflow as tf
from sklearn.model_selection import train_test_split
seed = 0; np.random.seed(seed)

<h1>>> Regular data

<h2>Import and split data

In [56]:
DATA = np.loadtxt('data/lol_dataset.csv', delimiter=',', skiprows=1)
X = DATA[:, :-1]
y = DATA[:, -1]
X_train, X_other, y_train, y_other = train_test_split(X, y, train_size=0.7, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=seed)

<h2>Hyperparameter tuning

<h4>Define dummy model for tuning

In [50]:
def test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs):
    first, second, third = units_per_layer

    if dropout_rate:
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(first, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(second, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(third, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])
    else:
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(first, activation='relu'),
            tf.keras.layers.Dense(second, activation='relu'),
            tf.keras.layers.Dense(third, activation='relu'),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy']
        )

    history = model.fit(
        X_train, 
        y_train, 
        batch_size=batch_size, 
        epochs=num_epochs, 
        validation_data=(X_val, y_val),
        verbose=0
        )
    
    accuracy = round(history.history["val_accuracy"][-1], 5)
    acc_len = len(str(accuracy))
    num_spaces = 7-acc_len 
    if accuracy >= 0.90:
        print(f"VAL ACCURACY: {round(accuracy, 5)}{" "*num_spaces}  |  units_per_layer: {units_per_layer}, dropout_rate: {dropout_rate}, learning_rate: {learning_rate}, batch_size: {batch_size}, num_epochs: {num_epochs}")

<h4>Test parameters

In [51]:
layers = [[128, 128, 64], [128, 64, 64], [64, 64, 64], [128, 64, 32], [128, 32, 32]]
dropout_rates = [None, 0.2]
learning_rates = [0.0001, 0.001, 0.01] # i tried also 0.1, 1.0 and 10.0 but these were consistently worse
batch_sizes = [64, 128]
nums_epochs = [5, 10, 15]

print("Architectures with an accuracy of 0.90 or higher...")
for units_per_layer in layers:
    for dropout_rate in dropout_rates:
        for learning_rate in learning_rates:
            for batch_size in batch_sizes:
                for num_epochs in nums_epochs:
                    test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs)

Architectures with an accuracy of 0.90 or higher...
VAL ACCURACY: 0.90176  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.92433  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.93445  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.92068  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.94756  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.9454   |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 128, num_epochs: 15
VAL ACCURACY: 0.90591  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 128, num_epochs: 10
VAL ACCURACY: 0.90707

<h2>Define and save the regular model

<h4>The best combination - VAL ACCURACY: 0.96117  |  units_per_layer: [128, 32, 32], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 15

In [57]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history = model.fit(
    X_train, 
    y_train, 
    batch_size=64, 
    epochs=15, 
    validation_data=(X_val, y_val),
    verbose=0
)

model.save('models/lol_model.keras')

<h1>>> Normalized data

<h2>Import and split data

In [58]:
DATA = np.loadtxt('data/lol_dataset_normalized.csv', delimiter=',', skiprows=1)
X = DATA[:, :-1]
y = DATA[:, -1]
X_train, X_other, y_train, y_other = train_test_split(X, y, train_size=0.7, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=seed)

<h2>Hyperparameter tuning

In [54]:
layers = [[128, 128, 64], [128, 64, 64], [64, 64, 64], [128, 64, 32], [128, 32, 32]]
dropout_rates = [None, 0.2]
learning_rates = [0.0001, 0.001, 0.01] # i tried also 0.1, 1.0 and 10.0 but these were consistently worse
batch_sizes = [64, 128]
nums_epochs = [5, 10, 15]

print("Architectures with an accuracy of 0.90 or higher...")
for units_per_layer in layers:
    for dropout_rate in dropout_rates:
        for learning_rate in learning_rates:
            for batch_size in batch_sizes:
                for num_epochs in nums_epochs:
                    test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs)

Architectures with an accuracy of 0.90 or higher...
VAL ACCURACY: 0.90856  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 15
VAL ACCURACY: 0.90989  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.92483  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.9386   |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 128, num_epochs: 15
VAL ACCURACY: 0.91537  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.924    |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.96515  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.9303   | 

<h2>Define and save the normalized model

<h4>The best combination -  VAL ACCURACY: 0.96416  |  units_per_layer: [64, 64, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 64, num_epochs: 15

In [59]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history = model.fit(
    X_train, 
    y_train, 
    batch_size= 64, 
    epochs=15, 
    validation_data=(X_val, y_val),
    verbose=0
)

model.save('models/lol_model_normalized.keras')